## Six-hump Camelback function

The objective is

$$

f(x, y) = \left(4 - 2.1x^2 + \frac{x^4}{3}\right) x^2 + xy + \left(-4 + 4y^2\right) y^2.

$$

We visualize the surface and the L-BFGS iterates starting from $(x_0, y_0) = (-1.8, -0.8)$. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.optimize import minimize


def sixhump_with_grad(x: np.ndarray):
    x0, x1 = x
    term1 = (4 - 2.1 * x0**2 + (x0**4) / 3.0) * x0**2
    term2 = x0 * x1
    term3 = (-4 + 4 * x1**2) * x1**2
    value = term1 + term2 + term3
    grad_x0 = 8 * x0 - 8.4 * x0**3 + 2 * x0**5 + x1
    grad_x1 = x0 - 8 * x1 + 16 * x1**3
    return value, np.array([grad_x0, grad_x1])


def sixhump_value(x: np.ndarray):
    return sixhump_with_grad(x)[0]


def sixhump_grad(x: np.ndarray):
    return sixhump_with_grad(x)[1]


# Grid for surface
x = np.linspace(-2, 0.5, 150)
y = np.linspace(-1, 1, 150)
xg, yg = np.meshgrid(x, y)
zg = sixhump_value(np.array([xg, yg]))

# L-BFGS-B from starting point (-1.8, -0.8)
x0 = np.array([-1.8, -0.8], dtype=float)
val0, grad0 = sixhump_with_grad(x0)
logs = [
    {
        "iter": -1,
        "x": float(x0[0]),
        "y": float(x0[1]),
        "f": float(val0),
        "grad_norm": float(np.linalg.norm(grad0)),
    }
]


def callback(xk: np.ndarray):
    val, grad = sixhump_with_grad(xk)
    logs.append(
        {
            "iter": len(logs),
            "x": float(xk[0]),
            "y": float(xk[1]),
            "f": float(val),
            "grad_norm": float(np.linalg.norm(grad)),
        }
    )


result = minimize(
    sixhump_value,
    x0=x0,
    method="L-BFGS-B",
    jac=sixhump_grad,
    callback=callback,
)

# Add final point to logs if not captured
val_final, grad_final = sixhump_with_grad(result.x)
if (
    not logs
    or logs[-1]["x"] != float(result.x[0])
    or logs[-1]["y"] != float(result.x[1])
):
    logs.append(
        {
            "iter": len(logs),
            "x": float(result.x[0]),
            "y": float(result.x[1]),
            "f": float(val_final),
            "grad_norm": float(np.linalg.norm(grad_final)),
        }
    )

print("Optimization log (iteration, x, y, f, ||grad||):")
for entry in logs:
    print(
        f"{entry['iter']:3d}: x={entry['x']:+.4f}, y={entry['y']:+.4f}, "
        f"f={entry['f']:+.6f}, grad_norm={entry['grad_norm']:.3e}"
    )

# Trajectory arrays
path = np.array([[e["x"], e["y"]] for e in logs])
path_z = sixhump_value(path.T)

# Surface only
fig1 = plt.figure(figsize=(6, 8))
ax1 = fig1.add_subplot(111, projection="3d")
ax1.plot_surface(
    xg, yg, zg, cmap=plt.colormaps.get_cmap("viridis"), linewidth=0, alpha=1.0
)
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_zlabel("f(x, y)")
ax1.set_title("Six-hump Camelback surface")
ax1.view_init(elev=30, azim=30)
plt.tight_layout()
plt.show()

# Surface with L-BFGS path and arrows
fig2 = plt.figure(figsize=(6, 8))
ax2 = fig2.add_subplot(111, projection="3d")
ax2.plot_surface(
    xg, yg, zg, cmap=plt.colormaps.get_cmap("viridis"), linewidth=0, alpha=0.7
)
ax2.plot(
    path[:, 0],
    path[:, 1],
    path_z,
    color="crimson",
    marker="o",
    markersize=3,
    linewidth=2,
    label="L-BFGS path",
)

# Draw arrows along path
if len(path) > 1:
    dx = np.diff(path[:, 0])
    dy = np.diff(path[:, 1])
    dz = np.diff(path_z)
    ax2.quiver(
        path[:-1, 0],
        path[:-1, 1],
        path_z[:-1],
        dx,
        dy,
        dz,
        color="black",
        arrow_length_ratio=0.2,
        linewidth=1,
    )

ax2.set_xlabel("x")
ax2.set_ylabel("y")
ax2.set_zlabel("f(x, y)")
ax2.set_title("L-BFGS path on Six-hump Camelback")
ax2.legend(loc="upper right")
ax2.view_init(elev=30, azim=30)
plt.tight_layout()
plt.show()
